# ****CATATAN KETIGA KOMPETISI****

---

#### ***SPESIFIK: STUDI KASUS***

Pada lomba ini, tim berperan sebagai analis risiko di PT. Investion dan harus ***membangun Early Warning System untuk memprediksi volatilitas realisasi IHSG 5 hari ke depan***. Modelnya wajib dapat dijelaskan (XAI = Explainable Artificial Intelligence), pipeline-nya wajib bebas kebocoran data, dan hasilnya wajib dibandingkan dengan baseline naif.

*Kata Kunci: Volatilitas, XAI*

- Volatilitas: merujuk pada ukuran statistik yang menunjukkan seberapa besar dan seberapa cepat nilai suatu data berubah atau berfluktuasi dalam periode waktu tertentu.
- XAI: serangkaian proses, metode, dan teknologi yang memungkinkan manusia untuk memahami, menafsirkan, dan mempercayai hasil atau keputusan yang dibuat oleh algoritma machine learning.

---

**1. Apa sebenarnya yang diminta?**

Studi kasus ini bukan meminta prediksi naik atau turunnya IHSG. Yang diminta adalah prediksi seberapa liar IHSG bergerak dalam 5 hari ke depan. Dalam manajemen risiko, besarnya gejolak menentukan keputusan seperti batas eksposur dan cadangan modal. Inilah arti “Sistem Peringatan Dini”: model memberi sinyal sebelum pasar bergolak.

Ada dua syarat tambahan dari direksi. Pertama, model harus bisa dijelaskan (XAI). Kedua, pipeline harus bersih dari kebocoran data masa depan

**2. Variabel Target Y**

Y adalah realized volatility, yaitu simpangan baku return harian IHSG dalam jendela 5 hari bursa. Hasil pemeriksaan menunjukkan Y di hari t sangat mirip (bukan identik) dengan simpangan baku return IHSG hari t+1 sampai t+5. Ini menunjukkan Y itu forward-looking.

Empat sifat Y yang wajib dipahami:

1. Miring ke kanan. Hari tenang banyak, krisis jarang tetapi ekstrem. Ini berpengaruh pada pemilihan loss dan evaluasi.
2. Volatility clustering. Hari bergejolak cenderung diikuti hari bergejolak (Mandelbrot, 1963; Engle, 1982). Itulah alasan masa lalu informatif.
3. Jendela yang tumpang tindih. Y hari t dan t+1 berbagi 4 dari 5 return. Akibatnya autokorelasi lag-1 mencapai 0,88 dan lag-5 turun ke 0,45. Dua konsekuensinya: jumlah sampel efektif jauh lebih kecil dari 3.907 baris, dan random split sangat bocor.
4. Artefak libur. Pada hari libur nilai IHSG di-forward-fill, sehingga return = 0 dan Y bisa = 0 palsu (16 baris).

**3. Arti Tiap Kolom dan Perannya**

1. Date
- *Arti*: Hari Senin - Jumat (2010 - 2026)
- *Mengapa relevan untuk volatilitas*: Urutan waktu adalah segalanya
- *Catatan penting*: Libur diisi nilai kemarin

2. IHSG
- *Arti*: Level indeks saham domestik
- *Mengapa relevan untuk volatilitas*: Dasar perhitungan Y
- *Catatan Penting*: Level non-stasioner, pakai return

3. SP500
- *Arti*: Indeks saham AS
- *Mengapa relevan untuk volatilitas*: Proksi kondisi pasar global dan spillover
- *Catatan Penting*: Naik terus dalam jangka panjang

4. VIX
- *Arti*: Ekspektasi volatilitas S&P500 (fear gauge; Whaley, 2000)
- *Mengapa relevan untuk volatilitas*: Sinyal ketakutan global
- *Catatan Penting*: Level juga informatif

5. USDIDR
- *Arti*: Rupiah per USD
- *Mengapa relevan untuk volatilitas*: Tekanan modal asing keluar
- *Catatan Penting*: Dua anomali (888,11 dan 892,0)

6. OIL
- *Arti*: Harga minyak (USD/Barel)
- *Mengapa relevan untuk volatilitas*: Guncangan komoditas dan inflasi
- *Catatan Penting*: Ada nilai negatif (−37,63)

7. GOLD
- *Arti*: Harga emas (USD/troy oz)
- *Mengapa relevan untuk volatilitas*: Aset safe haven
- *Catatan Penting*: 94% data uji di luar rentang train

8. CPI
- *Arti*: Inflasi Indonesia
- *Mengapa relevan untuk volatilitas*: Tekanan kebijakan moneter
- *Catatan Penting*: Hanya 17 nilai unik, berubah tiap 1 Januari

9. BI_rate
- *Arti*: Suku bunga acuan Bank Indonesia
- *Mengapa relevan untuk volatilitas*: Kebijakan moneter domestik
- *Catatan Penting*: Berubah jarang (langkah)

10. US_rate
- *Arti*: Federal Funds Rate
- *Mengapa relevan untuk volatilitas*: Pembanding suku bunga global
- *Catatan Penting*: Berubah jarang (langkah)

11. Target_Volatilitas_Y
- *Arti*: realized volatility
- *Mengapa relevan untuk volatilitas*: Target
- *Catatan Penting*: NaN untuk 2025 sampai Mei 2026

>Catatan Penting

1. Frekuensi campur. Enam kolom harian, tiga kolom (CPI, BI_rate, US_rate) berubah jarang. Pada kolom harian, nilai level hampir tidak informatif untuk volatilitas, sedangkan perubahan dan guncangannya justru informatif.

2. CPI. Berubah tiap 1 Januari dan nilainya mirip inflasi tahunan (%), bukan indeks seperti di kamus data. Nilai tahunan baru diketahui setelah tahun berakhir, jadi memakainya tanpa jeda berisiko look-ahead. Perlu diputuskan perlakuannya dan menjelaskannya.

3. Waktu perubahan suku bunga. Pada beberapa perubahan awal di train, nilai berganti di awal bulan. Di data uji, perubahannya jatuh di pertengahan bulan (mis. 2025-01-15). Ini baru dugaan, dan layak dicek karena bisa menandakan cara pencatatan yang berbeda.

4. Zona waktu. S&P500 dan VIX hari t tutup setelah IHSG hari t tutup, tetapi sebelum IHSG buka di t+1. Karena Y_t mengukur return mulai t+1, memakai SP500/VIX hari t sah (bukan kebocoran), dengan asumsi prediksi dibuat setelah semua pasar tutup. Asumsi ini harus tertulis jelas di Business Understanding.

**4. Strategi Fitur**

Mulai dari kerangka berpikirnya. Y = f(informasi yang diketahui hingga hari t). Fitur yang baik harus memenuhi empat syarat:

1. punya hipotesis ekonomi dengan arah pengaruh yang diharapkan,
2. tersedia pada hari t,
3. stasioner (tidak mengandalkan level yang terus naik),
4. punya nilai tambah yang dibuktikan di atas fitur volatilitas masa lalu.

>Syarat keempat sering terlupakan. Karena Y sangat persisten, hampir semua fitur akan tampak berkorelasi dengan Y lewat volatilitas yang tertangkap bersama. Fitur makro baru bernilai jika ia menambah akurasi di luar volatilitas historis IHSG, dan bukti seperti itulah yang membuat nilai EDA & Logic (20%) tinggi.

**5. Strategi Validasi**

Pikirkan seperti ini: model harus menjawab pertanyaan nyata bagi direksi, yaitu “dengan informasi sampai hari ini, volatilitas 5 hari ke depan berapa?” Maka validasi harus meniru situasi itu.

- Urutan waktu. Latih di masa lalu, uji di masa setelahnya. Jangan pernah shuffle.
- Purge/embargo ≥ 5 hari. Label di akhir train memakai return yang sudah masuk ke periode validasi. Beri jarak minimal 5 hari bursa antara akhir train dan awal validasi (López de Prado, 2018).
- Walk-forward beberapa lipatan. Validasi tunggal 2023–2024 kebetulan periode tenang (rata-rata Y 0,0053 dan 0,0070), sedangkan data uji turbulen. Pastikan ada lipatan yang mencakup masa stres, seperti 2020, agar kita tahu perilaku model saat krisis.
- Fit hanya di train. Scaler, imputasi, dan statistik apa pun dipelajari dari data latih dan baru diterapkan ke validasi dan uji.
- Keputusan yang harus Anda ambil: apakah model final dilatih ulang pada train+validasi dengan hiperparameter terbaik sebelum memprediksi data uji. Praktik ini lazim, tetapi alasannya harus tertulis

**6. Strategi Model**

Alur yang disarankan: baseline sederhana dulu, lalu model utama, lalu buktikan keunggulannya.

- Baseline. Naif (std 5 hari ke belakang, sesuai soal), rata-rata historis, dan HAR (regresi linear atas volatilitas 5/22 hari). GARCH(1,1) opsional sebagai pembanding klasik (Bollerslev, 1986).
- Model utama. Gradient boosting (LightGBM atau XGBoost). Alasannya: menangkap nonlinearitas dan interaksi (mis. VIX tinggi dan rupiah melemah), cocok untuk data tabular berukuran ribuan baris, dan SHAP versi pohon (TreeSHAP) memberi penjelasan eksak dan cepat (Lundberg et al., 2020).
- Mengapa bukan LSTM. Data hanya ±3,9 ribu baris dengan sampel efektif lebih kecil lagi, tuning mahal, mudah overfit, dan sulit dijelaskan. Untuk direksi yang menolak black-box, itu kelemahan.
- Tuning. Cari dengan Optuna (Akiba et al., 2019) memakai validasi walk-forward. Gunakan regularisasi kuat (kedalaman dangkal, learning rate kecil, early stopping) karena data kecil dan noisy.
- Transformasi target. Karena Y miring ke kanan, mencoba model pada skala log(Y) layak diuji. Skor akhir tetap dihitung pada skala asli. Waspadai Y = 0 (artefak libur) pada transformasi log.
- Metrik. RMSE dan MAE sesuai soal. Pertimbangkan juga menyajikan hasil per rezim (tenang vs krisis), karena rata-rata bisa menutupi kegagalan di saat paling penting.

>Satu ide opsional yang menambah nilai bisnis: ubah keluaran regresi menjadi alarm. Ambil ambang batas (misalnya persentil ke-90 dari data latih saja), lalu laporkan seberapa sering model berhasil memberi peringatan sebelum lonjakan. Direksi memahami “alarm berbunyi” lebih mudah daripada RMSE.

**7. Strategi XAI**

1. Global. SHAP summary/beeswarm dan bar untuk melihat fitur mana yang paling menentukan, plus dependence plot untuk melihat bentuk hubungannya.
2. Lokal. Pilih tanggal dengan prediksi volatilitas sangat tinggi, tampilkan waterfall, lalu terjemahkan ke bahasa bisnis: fitur ekonomi apa yang mendorong lonjakan itu.
3. Analisis kesalahan. Cari tanggal di mana model meleset jauh, bandingkan SHAP-nya dengan kondisi tipikal, lalu jelaskan penyebabnya dengan bukti, bukan dugaan.
4. Kehati-hatian. SHAP menjelaskan model, bukan sebab-akibat di dunia nyata. Fitur yang berkorelasi tinggi membagi kontribusi, jadi pertimbangkan pengelompokan fitur per keluarga. Menyebut keterbatasan ini di Q&A menunjukkan kedewasaan analisis.

Business Understanding → apa keputusan yang didukung, apa arti "berhasil"

Data Understanding     → kualitas data, rekonstruksi Y, pola, anomali, pergeseran rezim

Data Preparation       → bersihkan, fitur, split + purge, bukti anti-leakage

Modeling               → baseline → model utama → tuning

Evaluation + XAI       → bukti unggul, per rezim, global/lokal/kesalahan

Konsolidasi            → 1 notebook + PPT + latihan Q&A

---

Lain2

Struktur: 4.266 baris, 11 kolom, 2010-01-04 sampai 2026-05-22, hari kerja saja, tanpa missing value kecuali Y. Y kosong pada 359 baris (2025: 258, 2026: 101), jadi itulah data uji. Y valid terakhir jatuh pada 2024-12-24